# CorefInst — Inference & Evaluation

Run this notebook after training is complete.  
Assumes model weights are saved on Drive at `DRIVE_ROOT/model_output/final`.

Steps:
1. Mount Drive & set paths
2. Install dependencies
3. Clone / pull code
4. Extract dataset (needed for gold evaluation)
5. Patch config with Drive paths
6. Run inference
7. View results vs baselines


## Step 1 — Mount Drive & set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Edit these if your layout differs ────────────────────────────────────────
DRIVE_ROOT   = '/content/drive/MyDrive/CorefInst'
DATA_TAR     = f'{DRIVE_ROOT}/transmucores_data.tar.gz'
GITHUB_REPO  = 'https://github.com/pradneshfernandez/coreference-resolution'
PROJECT_DIR  = '/content/corefinst'
CHECKPOINT   = f'{DRIVE_ROOT}/model_output/final'
OUTPUT_DIR   = f'{DRIVE_ROOT}/inference_output'
DATA_DIR     = f'{DRIVE_ROOT}/transmucores_data'
# ─────────────────────────────────────────────────────────────────────────────

import os
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f'Checkpoint : {CHECKPOINT}')
print(f'Output dir : {OUTPUT_DIR}')
print(f'Checkpoint exists: {os.path.isdir(CHECKPOINT)}')


## Step 2 — Install dependencies

In [ ]:
!pip install -q 'unsloth[colab-new]' 2>/dev/null || pip install -q unsloth
!pip install -q transformers>=4.44.0 datasets>=2.20.0 peft>=0.12.0 \
               trl>=0.10.0 bitsandbytes>=0.43.0 accelerate>=0.33.0 scipy PyYAML
print('Done.')


## Step 3 — Get project code

In [ ]:
import os
if os.path.isdir(f'{PROJECT_DIR}/.git'):
    !cd {PROJECT_DIR} && git pull
else:
    !git clone {GITHUB_REPO} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
!pip install -q -e . --quiet
!git log --oneline -3


## Step 4 — Extract dataset to Drive (needed for gold evaluation)

Skips if already extracted.

In [ ]:
import os, tarfile

if os.path.isdir(DATA_DIR):
    print(f'Dataset already at {DATA_DIR}')
elif os.path.exists(DATA_TAR):
    print(f'Extracting to {DATA_DIR} …')
    with tarfile.open(DATA_TAR) as tf:
        tf.extractall(DRIVE_ROOT)
    for sub in ['mujadia_conll', 'onto_notes', 'litbank_train', 'litbank_val', 'litbank_test']:
        gz = os.path.join(DATA_DIR, f'{sub}.tar.gz')
        if os.path.exists(gz):
            with tarfile.open(gz) as tf:
                tf.extractall(DATA_DIR)
    print('Done.')
else:
    print(f'ERROR: {DATA_TAR} not found. Upload transmucores_data.tar.gz to {DRIVE_ROOT}')


## Step 5 — Patch config with Drive paths

In [ ]:
import yaml, os, torch

# Detect GPU and pick the right preset
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1e9
    gpu_name = props.name.lower()
    if 'h100' in gpu_name:   PRESET = 'h100'
    elif vram_gb >= 35:       PRESET = 'a100'
    elif vram_gb >= 20:       PRESET = 'l4'
    else:                     PRESET = 't4'
else:
    PRESET = 'cpu'

import shutil
preset_path = os.path.join(PROJECT_DIR, f'configs/{PRESET}.yaml')
config_path = os.path.join(PROJECT_DIR, 'config.yaml')
shutil.copy(preset_path, config_path)

with open(config_path) as f:
    cfg = yaml.safe_load(f)

cfg['data']['root']            = DATA_DIR
cfg['data']['output_dir']      = f'{DRIVE_ROOT}/processed_data'
cfg['training']['output_dir']  = f'{DRIVE_ROOT}/model_output'
cfg['inference']['output_dir'] = OUTPUT_DIR

with open(config_path, 'w') as f:
    yaml.dump(cfg, f)

print(f'Preset  : {PRESET}')
print(f'data.root       → {cfg["data"]["root"]}')
print(f'inference.dir   → {cfg["inference"]["output_dir"]}')


## Step 6 — Run inference

Processes all 740 test documents. Expect ~20–30 min on A100.

In [ ]:
!python scripts/run_inference.py \
    --config config.yaml \
    --split test \
    --checkpoint {CHECKPOINT} \
    --output_dir {OUTPUT_DIR}


## Step 7 — Results vs baselines

In [ ]:
# Model scores
!python analysis/analyse_results.py --results_json {OUTPUT_DIR}/results.json


In [ ]:
# Baseline comparison (all-singletons, all-one-cluster, MFE)
!python analysis/baseline.py --config config.yaml --split test


In [ ]:
# Verify all output files exist on Drive
import os
checks = [
    f'{OUTPUT_DIR}/results.json',
    f'{DRIVE_ROOT}/processed_data/test.jsonl',
    f'{CHECKPOINT}/adapter_config.json',
]
for path in checks:
    status = '✓' if os.path.exists(path) else '✗ MISSING'
    print(f'{status}  {path}')
